# Week 11 Pair Programming: Build a Tiny App

## Goal
Combine ONE traditional ML model (Weeks 1-9) with ONE generative model (Week 10-11) into a tiny working app.

## Setup
Find a partner. Together, pick ONE app idea. Build it in 15 minutes. Demo it briefly to the class.

## Time
20 minutes (15 build + 5 demo)

## Continuity
This integrates everything from the course — sklearn / Keras / Hugging Face / generative — into one pipeline.

## App ideas (pick one)

1. **Sentiment-aware response generator** — classify customer feedback sentiment (HF), generate appropriate response (LLM)
2. **Topic-aware summarizer** — classify article topic (HF), tailor summary length to topic
3. **Image-to-prompt-to-image** — classify input image (Week 9 transfer learning), use predicted class as Stable Diffusion prompt for stylized variations
4. **Cluster describer** — K-Means cluster (Week 5) on a small dataset, ask LLM to describe each cluster
5. **Code-from-description** — describe what code should do, ask LLM to generate it, run it, show output
6. **Resume scorer** — extract entities (HF NER), score against job description (LLM judgment), generate cover letter (LLM)
7. **Your own idea** — combine any 2 models in an interesting way

## Setup

In [ ]:
import torch
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM, set_seed

set_seed(42)
# Runs on GPU, Apple Silicon (MPS), or CPU.
DEVICE = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')

## Reusable helper: load LLM

In [ ]:
MODEL = 'microsoft/Phi-3-mini-4k-instruct'
# MODEL = 'Qwen/Qwen2.5-0.5B-Instruct'  # for slower machines / less memory

llm_tokenizer = AutoTokenizer.from_pretrained(MODEL)
llm_model = AutoModelForCausalLM.from_pretrained(
    MODEL,
    dtype=torch.float16 if DEVICE in ('cuda', 'mps') else torch.float32,  # v5: 'dtype' (was 'torch_dtype')
    device_map='auto' if DEVICE == 'cuda' else None,
    # No trust_remote_code: v5 has native Phi-3/Qwen support; the old bundled code crashes on v5.
)
if DEVICE != 'cuda':          # mps/cpu need an explicit move; cuda is placed by device_map='auto'
    llm_model = llm_model.to(DEVICE)

def llm_generate(prompt, max_new_tokens=120, temperature=0.7):
    messages = [{'role': 'user', 'content': prompt}]
    # v5: apply_chat_template returns a BatchEncoding — use return_dict=True + generate(**inputs).
    inputs = llm_tokenizer.apply_chat_template(
        messages, return_tensors='pt', add_generation_prompt=True, return_dict=True
    ).to(DEVICE)
    with torch.no_grad():
        outputs = llm_model.generate(
            **inputs, max_new_tokens=max_new_tokens, temperature=temperature,
            top_p=0.9, do_sample=True, pad_token_id=llm_tokenizer.eos_token_id
        )
    return llm_tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

## Example: App #1 — Sentiment-aware response generator

**Pipeline:** customer feedback → HF sentiment → LLM response generator (different prompt based on sentiment)

Run this as a template, then adapt to your chosen app idea.

In [ ]:
# Step 1: HF sentiment classifier (Week 10 tool)
sentiment = pipeline('sentiment-analysis')

# Step 2: define the app pipeline
def respond_to_customer(message):
    # Classify
    result = sentiment(message)[0]
    mood = result['label']
    confidence = result['score']
    
    # Generate appropriate response
    if mood == 'NEGATIVE':
        prompt = (
            f'A customer wrote this complaint: "{message}"\n\n'
            f'Write a brief, sincere apology that acknowledges their frustration '
            f'and offers to help. Maximum 3 sentences.'
        )
    else:
        prompt = (
            f'A customer wrote this positive feedback: "{message}"\n\n'
            f'Write a brief, warm thank-you message. Maximum 2 sentences.'
        )
    
    response = llm_generate(prompt, max_new_tokens=80, temperature=0.5)
    return {
        'sentiment': mood,
        'confidence': confidence,
        'response': response
    }

# Demo
test_messages = [
    'My order arrived broken and customer service ignored my emails.',
    'I just received my purchase and I am so impressed with the quality!',
    'The product is fine but shipping took longer than expected.',
]

for msg in test_messages:
    print(f'\nCustomer: {msg}')
    result = respond_to_customer(msg)
    print(f'  Sentiment: {result["sentiment"]} ({result["confidence"]:.2f})')
    print(f'  Response: {result["response"]}')

## YOUR TURN: build YOUR app

Pick an idea from the list above (or invent one). Adapt the template. Get something working in the next ~10 minutes.

**Constraints:**
- Must combine at least 2 models (one traditional ML + one generative)
- Must produce useful output for at least 3 example inputs
- Doesn't need to be polished — proof of concept is enough

In [ ]:
# Build your app here

# Step 1: load the traditional ML model you'll use
# Examples:
# - pipeline('sentiment-analysis')
# - pipeline('zero-shot-classification')
# - pipeline('ner', aggregation_strategy='simple')
# - keras.models.load_model(...) for a CNN/transfer learning model
# - sklearn model loaded via joblib

# Step 2: define your app pipeline
# def my_app(input_data):
#     ...

# Step 3: test on 3+ example inputs
# for example in examples:
#     output = my_app(example)
#     print(output)

## Demo (5 min)

When time's up, instructor will pick 2-3 pairs to demo their app. Show:
1. **What does it do?** (one sentence)
2. **Which models did you combine?**
3. **One input → output example**
4. **One thing that surprised you**

## Reflection

1. **What was easier than expected?**
2. **What was harder than expected?**
3. **If you had another hour, what would you add?**
4. **What would you NOT trust this app for in production?** (limits)
5. **In 2021, before ChatGPT, how would you have built this?** (Answer: you wouldn't have. Most of these patterns required LLM-level capability that didn't exist outside research labs.)

## Course closing thought

What you just built — combining 2+ ML models in 30 lines of code, using pretrained foundation models — was impossible 4 years ago. You've covered the entire trajectory of modern ML in 11 weeks.

**Go build things.**